<a href="https://colab.research.google.com/github/elijhagordon000/COMBINING-SEQUENTIAL-AND-GRAPH-SIGNALS-FOR-PERSONALIZED-MOVIE-RECOMMENDATION/blob/main/notebooks/lightGCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

# 1. Set environment variables directly (please replace with the actual content of your kaggle.json)
os.environ['KAGGLE_USERNAME'] = "tseng097"
os.environ['KAGGLE_KEY'] = "KGAT_3db2448f50eb504ee50ecd5884dea751"

# 2. Download the dataset (the system will automatically fetch the environment variables set above)
!kaggle datasets download -d grouplens/movielens-20m-dataset

# 3. Unzip the dataset
!unzip -q movielens-20m-dataset.zip -d movielens20m


Dataset URL: https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset
License(s): unknown
100% 195M/195M [00:01<00:00, 106MB/s]



# Dataloading

In [ ]:
# 4. Read the data
movies = pd.read_csv("movielens20m/movie.csv")
ratings = pd.read_csv("movielens20m/rating.csv")
tags = pd.read_csv("movielens20m/tag.csv")

# 5. Verify the result
print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
import numpy as np

# 1. Read data (continuing with your previous variables)
ratings = pd.read_csv("movielens20m/rating.csv")

# To run smoothly on Colab, we randomly sample 500,000 data points for demo purposes (for full training, please use DataLoader for batches)
ratings = ratings.sample(n=500000, random_state=42)

# 2. Re-encode IDs (continuous numbering starting from 0)
user_mapping = {id: i for i, id in enumerate(ratings['userId'].unique())}
movie_mapping = {id: i for i, id in enumerate(ratings['movieId'].unique())}

ratings['user_idx'] = ratings['userId'].map(user_mapping)
ratings['movie_idx'] = ratings['movieId'].map(movie_mapping)

num_users = len(user_mapping)
num_movies = len(movie_mapping)
num_nodes = num_users + num_movies

print(f"Number of users: {num_users}, Number of movies: {num_movies}, Total nodes: {num_nodes}")

# 3. Create Edge Index (PyG's graph structure format [2, num_edges])
# Note: Movie IDs must be added to num_users to ensure node IDs do not conflict with user IDs
user_nodes = ratings['user_idx'].values
movie_nodes = ratings['movie_idx'].values + num_users

# Create bidirectional edges (User -> Movie and Movie -> User)
edge_index_u2m = np.vstack([user_nodes, movie_nodes])
edge_index_m2u = np.vstack([movie_nodes, user_nodes])
edge_index = np.hstack([edge_index_u2m, edge_index_m2u])

edge_index = torch.tensor(edge_index, dtype=torch.long)
print(f"Edge Index Shape: {edge_index.shape}")

Number of users: 106761, Number of movies: 13152, Total nodes: 119913
Edge Index Shape: torch.Size([2, 1000000])


In [ ]:
import torch
from torch_geometric.utils import train_test_split_edges


def split_data(edge_index, num_users):

    edge_index = edge_index.to('cpu')
    num_edges = edge_index.shape[1]
    perm = torch.randperm(num_edges)
    train_size = int(0.8 * num_edges)

    train_edges = edge_index[:, perm[:train_size]]
    test_edges = edge_index[:, perm[train_size:]]

    return train_edges, test_edges

train_edge_index, test_edge_index = split_data(edge_index, num_users)
print(f"training: {train_edge_index.shape[1]}, testing: {test_edge_index.shape[1]}")

training: 800000, testing: 200000


## LightGCN Model Training and Evaluation

In [ ]:
from torch_geometric.nn import LightGCN
import torch.optim as optim
from torch_geometric.utils import structured_negative_sampling

# Set device (GPU preferred)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
edge_index = edge_index.to(device)

# 1. Initialize the model
model = LightGCN(
    num_nodes=num_nodes,
    embedding_dim=64,   # Embedding dimension
    num_layers=3        # Number of message passing layers (usually 2-4 layers work best)
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training parameters
epochs = 100

print(f"Starting training on device: {device}")
model.train()
for epoch in range(epochs):
    optimizer.zero_grad()

    # 1. Sample from the current edge_index
    u, pos, neg = structured_negative_sampling(edge_index)

    # 2. Prepare edge_label_index: Combine positive and negative sample node pairs
    pos_edge_label_index = torch.stack([u, pos], dim=0)
    neg_edge_label_index = torch.stack([u, neg], dim=0)

    # Concatenate positive and negative samples (positive samples first, then negative samples) Shape: [2, 2 * num_edges]
    edge_label_index = torch.cat([pos_edge_label_index, neg_edge_label_index], dim=1)

    # 3. Make predictions with the model
    # The model calculates embeddings for each node and then computes dot product scores for each (user, item) pair in edge_label_index
    out = model(edge_index, edge_label_index)

    # 4. Split the prediction scores in half: the first half for positive samples, the second half for negative samples
    pos_rank, neg_rank = out.chunk(2)

    # 5. Calculate the BPR Loss for the recommendation system
    loss = model.recommendation_loss(
        pos_edge_rank=pos_rank,     # Predicted scores for positive samples
        neg_edge_rank=neg_rank,     # Predicted scores for negative samples
        node_id=edge_label_index.unique() # Tell the model which nodes participated in this round for L2 regularization calculation
    )

    # Backpropagation
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch+1:03d}, BPR Loss: {loss.item():.4f}")

Starting training on device: cuda
Epoch: 010, BPR Loss: 0.6930
Epoch: 020, BPR Loss: 0.6898
Epoch: 030, BPR Loss: 0.6779
Epoch: 040, BPR Loss: 0.6537
Epoch: 050, BPR Loss: 0.6182
Epoch: 060, BPR Loss: 0.5778
Epoch: 070, BPR Loss: 0.5408
Epoch: 080, BPR Loss: 0.5124
Epoch: 090, BPR Loss: 0.4929
Epoch: 100, BPR Loss: 0.4820


In [ ]:
# ==========================================
# Stage 1: Calculate recommendation scores using the trained model
# ==========================================
model.eval()
with torch.no_grad():
    # Get the final embeddings of all nodes aggregated by LightGCN
    all_embeddings = model.get_embedding(edge_index)

    # Separate User and Movie Embeddings
    user_embeddings = all_embeddings[:num_users]
    movie_embeddings = all_embeddings[num_users:]

    # Specify the user to recommend to (e.g., user_idx = 0)
    target_user_idx = 0
    target_user_emb = user_embeddings[target_user_idx]

    # Calculate scores for this user with all movies (dot product)
    scores = torch.matmul(movie_embeddings, target_user_emb)

    # Find the top 10 movie internal indices with the highest scores
    top10_movie_idx = torch.topk(scores, k=10).indices.cpu().numpy()

print(f"Internal indices of movies recommended to User {target_user_idx}: {top10_movie_idx}\n")


# ==========================================
# Stage 2: Convert internal indices to real movie names and print them
# ==========================================
# Create reverse mapping dictionary (Internal Index -> Original movieId)
reverse_movie_mapping = {idx: movie_id for movie_id, idx in movie_mapping.items()}

# Convert the numpy array output by the model to a Python List
top10_idx = top10_movie_idx.tolist()

# Convert back to original movieId
top10_movie_ids = [reverse_movie_mapping[idx] for idx in top10_idx]

# Map to movie names and print
print(f"🎬 Top 10 Movies Recommended to User {target_user_idx}:\n" + "-"*45)

for rank, movie_id in enumerate(top10_movie_ids, 1):
    # Find the movie information for that movieId from the movies DataFrame
    movie_info = movies[movies['movieId'] == movie_id]

    if not movie_info.empty:
        title = movie_info.iloc[0]['title']
        genres = movie_info.iloc[0]['genres']
        print(f"Top {rank:>2}: {title}")
        print(f"        (Genres: {genres})")
    else:
        print(f"Top {rank:>2}: Could not find data for movieId {movie_id}")

Internal indices of movies recommended to User 0: [  41  100  277   51 1053  699   95  105  846 1109]

🎬 Top 10 Movies Recommended to User 0:
---------------------------------------------
Top  1: Pulp Fiction (1994)
        (Genres: Comedy|Crime|Drama|Thriller)
Top  2: Forrest Gump (1994)
        (Genres: Comedy|Drama|Romance|War)
Top  3: Shawshank Redemption, The (1994)
        (Genres: Crime|Drama)
Top  4: Silence of the Lambs, The (1991)
        (Genres: Crime|Horror|Thriller)
Top  5: Jurassic Park (1993)
        (Genres: Action|Adventure|Sci-Fi|Thriller)
Top  6: Star Wars: Episode IV - A New Hope (1977)
        (Genres: Action|Adventure|Sci-Fi)
Top  7: Terminator 2: Judgment Day (1991)
        (Genres: Action|Sci-Fi)
Top  8: Braveheart (1995)
        (Genres: Action|Drama|War)
Top  9: Matrix, The (1999)
        (Genres: Action|Sci-Fi|Thriller)
Top 10: Schindler's List (1993)
        (Genres: Drama|War)


In [ ]:
def get_popularity_recommendations(train_edges, num_users, num_movies, k=10):

    item_nodes = train_edges[1, train_edges[1] >= num_users]
    unique_items, counts = torch.unique(item_nodes, return_counts=True)


    _, top_indices = torch.topk(counts, k=min(k, len(unique_items)))
    popular_items = unique_items[top_indices] - num_users

    return popular_items.tolist()

pop_recs = get_popularity_recommendations(train_edge_index, num_users, num_movies)

In [ ]:
import numpy as np

def evaluate(model, edge_index, test_edges, num_users, k=20):
    model.eval()
    with torch.no_grad():
        out = model.get_embedding(edge_index)
        user_emb, item_emb = out[:num_users], out[num_users:]

    recalls, ndcgs = [], []

    # For performance, we sample 1000 users for evaluation
    test_users = torch.unique(test_edges[0])
    # Filter to ensure we only consider user nodes (indices less than num_users)
    test_users = test_users[test_users < num_users]
    sample_users = test_users[torch.randperm(len(test_users))[:1000]]

    for u in sample_users:
        # 1. Get movies actually watched by this user in the test set
        ground_truth = test_edges[1, test_edges[0] == u] - num_users
        if len(ground_truth) == 0: continue

        # 2. LightGCN Recommendation
        scores = torch.matmul(item_emb, user_emb[u])
        _, top_k_idx = torch.topk(scores, k)
        recs = top_k_idx.cpu().numpy()

        # 3. Calculate Recall
        hits = np.isin(recs, ground_truth.cpu().numpy())
        recall = np.sum(hits) / len(ground_truth)
        recalls.append(recall)

        # 4. Calculate NDCG
        dcg = np.sum(hits / np.log2(np.arange(2, k + 2)))
        idcg = np.sum(1.0 / np.log2(np.arange(2, min(len(ground_truth), k) + 2)))
        ndcgs.append(dcg / idcg)

    return np.mean(recalls), np.mean(ndcgs)

# Execute after training
recall_score, ndcg_score = evaluate(model, train_edge_index.to(device), test_edge_index, num_users, k=10)

In [ ]:
print(f"\nLightGCN Evaluation (k=10):\nRecall@10: {recall_score:.4f}\nNDCG@10: {ndcg_score:.4f}")


LightGCN Evaluation (k=10):
Recall@10: 0.0361
NDCG@10: 0.0204


## BPR-MF Model(baseline) Training and Evaluation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# ==========================================
# 1. BPR-MF Model (Regularization changed to 'only for embeddings in the batch')
# ==========================================
class BPRMF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def forward(self, users, pos_items, neg_items):
        u_e, p_e, n_e = self.user_emb(users), self.item_emb(pos_items), self.item_emb(neg_items)
        pos_scores = (u_e * p_e).sum(dim=1)
        neg_scores = (u_e * n_e).sum(dim=1)
        # L2 regularization only for embeddings used in the current batch (key correction!)
        reg = (u_e.pow(2).sum() + p_e.pow(2).sum() + n_e.pow(2).sum()) / u_e.size(0)
        return pos_scores, neg_scores, reg

    def bpr_loss(self, pos_scores, neg_scores, reg, reg_lambda=1e-4):
        bpr = -torch.mean(F.logsigmoid(pos_scores - neg_scores))
        return bpr + reg_lambda * reg, bpr.item()

    def get_embedding(self, edge_index=None):
        return torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)

# ==========================================
# 2. Prepare clean training pairs (same as previous version)
# ==========================================
mask = train_edge_index[0] < num_users
train_u   = train_edge_index[0][mask].to(device)
train_pos = (train_edge_index[1][mask] - num_users).to(device)
print(f"✅ Training pairs: {train_u.size(0):,}")

# ==========================================
# 3. Train and record history
# ==========================================
mf_model = BPRMF(num_users, num_movies, embedding_dim=64).to(device)
mf_optimizer = optim.Adam(mf_model.parameters(), lr=0.005)  # Increased learning rate

epochs = 100
eval_every = 10

history = {"epoch": [], "loss": [], "bpr": [], "recall": [], "ndcg": []}

print("🚀 Training BPR-MF...")
for epoch in range(1, epochs + 1):
    mf_model.train()
    mf_optimizer.zero_grad()
    neg_idx = torch.randint(0, num_movies, (train_u.size(0),), device=device)
    ps, ns, reg = mf_model(train_u, train_pos, neg_idx)
    loss, bpr_only = mf_model.bpr_loss(ps, ns, reg, reg_lambda=1e-4)
    loss.backward()
    mf_optimizer.step()

    if epoch % eval_every == 0:
        r, n = evaluate(mf_model, train_edge_index.to(device),
                        test_edge_index, num_users, k=10)
        history["epoch"].append(epoch)
        history["loss"].append(loss.item())
        history["bpr"].append(bpr_only)
        history["recall"].append(r)
        history["ndcg"].append(n)
        print(f"Epoch {epoch:03d} | Total: {loss.item():.4f} | "
              f"BPR: {bpr_only:.4f} | Recall@10: {r:.4f} | NDCG@10: {n:.4f}")

✅ Training pairs: 400,054
🚀 Training BPR-MF...
Epoch 010 | Total: 0.6486 | BPR: 0.6483 | Recall@10: 0.0013 | NDCG@10: 0.0013
Epoch 020 | Total: 0.5770 | BPR: 0.5767 | Recall@10: 0.0028 | NDCG@10: 0.0019
Epoch 030 | Total: 0.4786 | BPR: 0.4781 | Recall@10: 0.0016 | NDCG@10: 0.0008
Epoch 040 | Total: 0.3653 | BPR: 0.3646 | Recall@10: 0.0044 | NDCG@10: 0.0018
Epoch 050 | Total: 0.2585 | BPR: 0.2575 | Recall@10: 0.0067 | NDCG@10: 0.0024
Epoch 060 | Total: 0.1742 | BPR: 0.1729 | Recall@10: 0.0057 | NDCG@10: 0.0029
Epoch 070 | Total: 0.1194 | BPR: 0.1179 | Recall@10: 0.0051 | NDCG@10: 0.0028
Epoch 080 | Total: 0.0869 | BPR: 0.0850 | Recall@10: 0.0076 | NDCG@10: 0.0044
Epoch 090 | Total: 0.0683 | BPR: 0.0663 | Recall@10: 0.0083 | NDCG@10: 0.0044
Epoch 100 | Total: 0.0563 | BPR: 0.0540 | Recall@10: 0.0054 | NDCG@10: 0.0025


### Model Comparison: LightGCN vs. BPRMF

Based on the training and evaluation results:

**LightGCN Evaluation (Recall@10 and NDCG@10):**
- **Recall@10:** 0.0361
- **NDCG@10:** 0.0204

**BPRMF Evaluation (Final Recall@10 and NDCG@10):**
- **Recall@10:** 0.0054
- **NDCG@10:** 0.0025

*(Note: The LightGCN metrics will be available after the preceding cell runs.)*

Comparing the final metrics, we can observe which model performed better in terms of recommending relevant items to users in the test set.